Ejemplos de código que incluye cuatro algoritmos de búsqueda de cadenas: búsqueda nativa, Knuth-Morris-Pratt (KMP), Boyer-Moore y Rabin-Karp

In [2]:
import random
import time
import hashlib

# Generar secuencia de ADN
def generar_secuencia_adn(longitud):
    bases = ['A', 'T', 'C', 'G']
    return ''.join(random.choice(bases) for _ in range(longitud))



# Búsqueda nativa (búsqueda secuencial)


## Ventajas:
- Simple y fácil de implementar.
- Funciona bien para textos cortos y patrones cortos.

## Inconvenientes:
- No es eficiente para textos largos o patrones largos.
- En el peor de los casos, puede tomar tiempo ($ O(m · (n-m+1))$), donde \(m\) es la longitud del patrón y \(n\) es la longitud del texto.


In [3]:
def native_search(dna_sequence, query_sequence):
    return dna_sequence.find(query_sequence)

# **Algoritmo de Knuth-Morris-Pratt (KMP)**


## Ventajas:
- No retrocede en el texto durante la búsqueda, lo que lo hace más eficiente.
- El tiempo de búsqueda es ($O(n)$) después del preprocesamiento.

## Inconvenientes:
- Requiere espacio adicional para almacenar el array de prefijos.
- El preprocesamiento puede tomar tiempo adicional.


In [4]:
# Algoritmo de Knuth-Morris-Pratt (KMP)
def kmp_search(dna_sequence, query_sequence):
    m = len(query_sequence)
    n = len(dna_sequence)
    lps = [0] * m
    j = 0


    compute_lps_array(query_sequence, m, lps)

    i = 0
    while i < n:
        if query_sequence[j] == dna_sequence[i]:
            i += 1
            j += 1

        if j == m:
            return i - j
            j = lps[j - 1]

        elif i < n and query_sequence[j] != dna_sequence[i]:
            if j != 0:
                j = lps[j - 1]
            else:
                i += 1

    return -1

def compute_lps_array(query_sequence, m, lps):
    length = 0
    i = 1
    lps[0] = 0  # lps[0] is always 0

    # the loop calculates lps[i] for i = 1 to m-1
    while i < m:
        if query_sequence[i] == query_sequence[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length-1]
                # Note that we do not increment i here
            else:
                lps[i] = 0
                i += 1



# **Algoritmo de Boyer-Moore**


## Ventajas:
- Muy eficiente para alfabetos grandes y patrones no triviales.
- En el mejor caso, puede ofrecer un tiempo de búsqueda de ($O(n/m)$), donde \(m\) es la longitud del patrón.

## Inconvenientes:
- El preprocesamiento puede ser complejo y tomar tiempo.
- Menos eficiente para patrones y textos cortos.


In [4]:
# Algoritmo de Boyer-Moore
def boyer_moore_search(dna_sequence, query_sequence):
    m = len(query_sequence)
    n = len(dna_sequence)

    # Preprocesamiento
    badChar = [-1] * 256
    bad_char_heuristic(query_sequence, m, badChar)

    s = 0
    while s <= n - m:
        j = m - 1

        while j >= 0 and query_sequence[j] == dna_sequence[s + j]:
            j -= 1

        if j < 0:
            return s
            s += m - badChar[ord(dna_sequence[s + m])]
        else:
            s += max(1, j - badChar[ord(dna_sequence[s + j])])

    return -1

def bad_char_heuristic(query_sequence, m, badChar):
    for i in range(m):
        badChar[ord(query_sequence[i])] = i



# **Algoritmo de Rabin-Karp**


## Ventajas:
- Funciona bien para múltiples búsquedas del mismo patrón en múltiples textos.
- Es útil para la detección de plagio y búsquedas de patrones en múltiples documentos.

## Inconvenientes:
- Dependiente del valor hash y del número primo elegido, puede tener colisiones.
- En el peor caso, el tiempo de búsqueda puede ser \(O(n \cdot m)\).


In [5]:
def rabin_karp_search(dna_sequence, query_sequence):
    m = len(query_sequence)
    n = len(dna_sequence)
    pattern_hash = hash_function(query_sequence, m)
    text_hash = hash_function(dna_sequence[:m], m)
    base = 256  # Base para el cálculo del hash (número de caracteres en el alfabeto ASCII)

    for s in range(n - m + 1):
        if pattern_hash == text_hash:
            match = True
            for j in range(m):
                if dna_sequence[s + j] != query_sequence[j]:
                    match = False
                    break
            if match:
                return s  # Patrón encontrado en el índice s

        # Rehash para la siguiente ventana de texto
        if s < n - m:
            text_hash = ((text_hash - ord(dna_sequence[s]) * pow(base, m - 1)) * base +
                         ord(dna_sequence[s + m])) % 101  # 101 es un número primo

    return -1  # Patrón no encontrado

def hash_function(string, length):
    hash_value = 0
    for i in range(length):
        hash_value = (hash_value * 256 + ord(string[i])) % 101  # 101 es un número primo
    return hash_value


# **Algoritmo Z**


## Ventajas:
- Eficiente para búsquedas de patrones repetidos en un texto, ya que preprocesa el patrón.
- Útil para operaciones de búsqueda en tiempo real.

## Inconvenientes:
- El preprocesamiento puede ser costoso en términos de tiempo para patrones largos.
- Utiliza espacio adicional para almacenar el array Z.


In [6]:
# Algoritmo de búsqueda Z
def z_search(dna_sequence, query_sequence):
    concat = query_sequence + "$" + dna_sequence
    n = len(concat)
    z = [0] * n

    # [L,R] make a window which matches with prefix of s
    l, r, k = 0, 0, 0
    for i in range(1, n):

        if i > r:
            l, r = i, i

            while r < n and concat[r - l] == concat[r]:
                r += 1
            z[i] = r - l
            r -= 1

        else:
            k = i - l

            if z[k] < r - i + 1:
                z[i] = z[k]

            else:
                l = i
                while r < n and concat[r - l] == concat[r]:
                    r += 1
                z[i] = r - l
                r -= 1

    for i in range(n):
        if z[i] == len(query_sequence):
            return i - len(query_sequence) - 1
    return -1



# **Algoritmo de Finite Automaton**


## Ventajas:
- El tiempo de búsqueda es \(O(n)\) después del preprocesamiento.
- Funciona bien para alfabetos grandes y patrones largos.

## Inconvenientes:
- El preprocesamiento puede ser costoso, ($O(m^3 \cdot \text{ALPHABET_SIZE})$) en el peor de los casos.
- Utiliza espacio adicional para almacenar la matriz de transición.


In [7]:
# Algoritmo de búsqueda de Finite Automaton
def fa_search(dna_sequence, query_sequence):
    NO_OF_CHARS = 256

    def build_transition_function(query_sequence, m, tf):
        x = 0
        for i in range(NO_OF_CHARS):
            tf[i][0] = 0
        tf[ord(query_sequence[0])][0] = 1

        for i in range(1, m):
            for j in range(NO_OF_CHARS):
                tf[j][i] = tf[j][x]

            tf[ord(query_sequence[i])][i] = i + 1

            if i < m:
                x = tf[ord(query_sequence[i])][x]

    m = len(query_sequence)
    n = len(dna_sequence)
    tf = [[0] * (m+1) for _ in range(NO_OF_CHARS)]

    build_transition_function(query_sequence, m, tf)

    state = 0
    for i in range(n):
        state = tf[ord(dna_sequence[i])][state]
        if state == m:
            return i - m + 1

    return -1

In [14]:
# Generar secuencia de ADN
longitud = 10000000
dna_sequence = generar_secuencia_adn(longitud)

# Secuencia de consulta
query_sequence = "ATAGG"



In [15]:
# Evaluar la eficiencia temporal de cada algoritmo
start_time = time.time()
position = native_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"Native Search - Position: {position}, Time: {end_time - start_time} seconds")

start_time = time.time()
position = kmp_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"KMP Search - Position: {position}, Time: {end_time - start_time} seconds")

start_time = time.time()
position = boyer_moore_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"Boyer-Moore Search - Position: {position}, Time: {end_time - start_time} seconds")

start_time = time.time()
position = rabin_karp_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"Rabin-Karp Search - Position: {position}, Time: {end_time - start_time} seconds")

start_time = time.time()
position = z_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"Z-Algorithm Search - Position: {position}, Time: {end_time - start_time} seconds")

start_time = time.time()
position = fa_search(dna_sequence, query_sequence)
end_time = time.time()
print(f"Finite Automaton Search - Position: {position}, Time: {end_time - start_time} seconds")


Native Search - Position: 3831, Time: 0.0 seconds
KMP Search - Position: 3831, Time: 0.0 seconds
Boyer-Moore Search - Position: 3831, Time: 0.0010306835174560547 seconds
Rabin-Karp Search - Position: 3831, Time: 0.0009949207305908203 seconds
Z-Algorithm Search - Position: 3831, Time: 1.0474653244018555 seconds
Finite Automaton Search - Position: 3831, Time: 0.0 seconds
